# Traffic Density Detection - Module A
Chạy detection module trực tiếp từ main.py

In [6]:
# Chạy detection với giới hạn frames - hiển thị kết quả + thời gian
import os
import sys
import time
import json

# Thêm project root - dùng đường dẫn tuyệt đối
project_root = r"d:\GIT REPO\trafffic-density-analysis-system\traffic-density-analysis-system"
sys.path.insert(0, project_root)

# Cấu hình
MAX_FRAMES = 100  # Giới hạn số frames để test nhanh

BASE_DIR = os.path.join(project_root, "detection")
camera_id = "CAM_01"
config_path = os.path.join(BASE_DIR, "configs_cameras", "cam_01.json")
MODEL_PATH = os.path.join(BASE_DIR, "pro_models", "yolov9_img960_ultimate.pt")
VIDEO_SOURCE = os.path.join(project_root, "traffictrim.mp4")

# Debug: kiểm tra đường dẫn
print(f"Project root: {project_root}")
print(f"Config exists: {os.path.exists(config_path)}")
print(f"Model exists: {os.path.exists(MODEL_PATH)}")
print(f"Video exists: {os.path.exists(VIDEO_SOURCE)}")

CONF_THRESHOLD = 0.5
TARGET_WIDTH = 960

print("🚀 Starting detection...")
print(f"   Max frames: {MAX_FRAMES}")
print(f"   Video: {VIDEO_SOURCE}")
print(f"   Model: {MODEL_PATH}")
print("=" * 50)

# Import components
from detection.camera_engine import CameraEngine
from detection.engine.frame_processor import FrameProcessor
from detection.engine.detector import Detector
from detection.engine.tracker import Tracker
from detection.engine.counter import VehicleCounter
from detection.engine.density_estimator import DensityEstimator
from detection.engine.zone_manager import ZoneManager
from detection.engine.event_generator import EventGenerator

# Load config
with open(config_path, encoding="utf-8") as f:
    camera_config = json.load(f)

# Initialize
camera = CameraEngine(VIDEO_SOURCE)
processor = FrameProcessor(target_width=TARGET_WIDTH)
detector = Detector(MODEL_PATH, conf_threshold=CONF_THRESHOLD)
tracker = Tracker(lost_track_buffer=10)
counter = VehicleCounter()
density_estimator = DensityEstimator()
event_generator = EventGenerator()
zone_manager = ZoneManager(camera_config["zones"])

print(f"✓ Initialized | FPS: {camera.get_fps()}")
print("-" * 50)

# Run detection
start_time = time.time()
frame_count = 0
results_history = []

try:
    while frame_count < MAX_FRAMES:
        ret, frame = camera.read()
        if not ret:
            print("📹 Video ended")
            break
        
        frame_count += 1
        
        # Process
        frame = processor.process(frame)
        detections = detector.detect(frame)
        tracks = tracker.update(detections, frame)
        
        # Update density
        density_estimator.update(tracks)
        traffic_density = density_estimator.get_density()
        
        # Count vehicles crossing zones
        for track in tracks:
            x1, y1, x2, y2 = track["bbox"]
            track_id = track["track_id"]
            vehicle_type = track["class_name"]
            
            cx = int((x1 + x2) // 2)
            cy_bottom = int(y2)
            
            if zone_manager.check_crossing(track_id, cx, cy_bottom):
                counter.count(vehicle_type)
        
        # Log every 20 frames
        if frame_count % 20 == 0:
            print(f"Frame {frame_count:3d} | Density: {traffic_density:6s} | Tracks: {len(tracks):2d} | Total: {sum(counter.get_totals().values())}")

except KeyboardInterrupt:
    print("⏹️ Interrupted")
finally:
    camera.release()

end_time = time.time()
elapsed = end_time - start_time

# Results
print("=" * 50)
print("📊 RESULTS")
print("=" * 50)
print(f"Frames processed: {frame_count}")
print(f"Execution time:   {elapsed:.2f} seconds")
print(f"FPS (processing): {frame_count/elapsed:.1f} fps")
print(f"\nVehicle counts:")
for vehicle, count in counter.get_totals().items():
    print(f"  {vehicle}: {count}")
print(f"  TOTAL: {sum(counter.get_totals().values())}")

Project root: d:\GIT REPO\trafffic-density-analysis-system\traffic-density-analysis-system
Config exists: True
Model exists: True
Video exists: True
🚀 Starting detection...
   Max frames: 100
   Video: d:\GIT REPO\trafffic-density-analysis-system\traffic-density-analysis-system\traffictrim.mp4
   Model: d:\GIT REPO\trafffic-density-analysis-system\traffic-density-analysis-system\detection\pro_models\yolov9_img960_ultimate.pt
Loaded model successfully from: d:\GIT REPO\trafffic-density-analysis-system\traffic-density-analysis-system\detection\pro_models\yolov9_img960_ultimate.pt
✓ Initialized | FPS: 30.0
--------------------------------------------------
Frame  20 | Density: MEDIUM | Tracks:  8 | Total: 2
Frame  40 | Density: MEDIUM | Tracks:  9 | Total: 2
Frame  60 | Density: MEDIUM | Tracks:  8 | Total: 3
Frame  80 | Density: MEDIUM | Tracks:  9 | Total: 6
Frame 100 | Density: MEDIUM | Tracks: 11 | Total: 6
📊 RESULTS
Frames processed: 100
Execution time:   443.48 seconds
FPS (processi